<a href="https://colab.research.google.com/github/shivangi221b/ASR-Fine-Tuning-Modules-and-RL-Adaptation-Analysis/blob/shivangi_initial_setup/NeMo_STT_Domain_Adaptation_SFT_%2B_RL_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeMo STT Domain Adaptation: SFT + RL Training Pipeline
Uses AfriSpeech-200 clinical subset for healthcare domain adaptation.

Two-stage training:
  - Stage 1: Supervised Fine-Tuning (SFT) with CTC loss
  - Stage 2: RL fine-tuning on top of SFT with reward injection

Datasets supported:
  - AfriSpeech-200 clinical (primary healthcare)
  - VoxPopuli English (judicial proxy) - for GCP full run
  - LibriSpeech clean (general baseline) - for GCP full run

Usage:
  # Smoke test on Colab (AfriSpeech clinical only)
  python nemo_afrispeech_training.py --stage both
  
  # GCP: Run SFT only
  python nemo_afrispeech_training.py --stage sft
  
  # GCP: Run RL on existing SFT checkpoint
  python nemo_afrispeech_training.py --stage rl --sft_checkpoint ./checkpoints/sft_model.nemo


In [1]:
pip install -q nemo_toolkit[asr]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.7/889.7 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.1/443.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.9 MB/s eta 0:0

In [2]:
pip install -q datasets soundfile librosa jiwer

In [1]:
import os
import json
import time
import logging
import math
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import types

import torch
import numpy as np
import soundfile as sf
import pandas as pd
from omegaconf import OmegaConf, DictConfig

# NeMo imports
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.models import EncDecCTCModelBPE

# PyTorch Lightning - CRITICAL: use lightning.pytorch for NeMo 2.x
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback

# Data utilities
from datasets import load_dataset, Audio

# WER computation
from jiwer import wer as compute_wer_jiwer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

[NeMo W 2026-03-31 23:30:44 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-03-31 23:30:47 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-03-31 23:30:47 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-03-31 23:30:47 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-03-31 23:30:47 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


# CONFIGURATION - EDIT THESE VALUES

In [2]:
# Model
NEMO_MODEL_NAME = "stt_en_conformer_ctc_small"  # ~13M params

# Dataset sizes (smoke test)
TRAIN_SAMPLES = 200   # Full GCP run: 1000
EVAL_SAMPLES = 100    # Full GCP run: 200

# Training hyperparameters
BATCH_SIZE = 8        # A100: 16, T4: 8
LEARNING_RATE = 1e-4

# Stage 1: SFT
SFT_EPOCHS = 2

# Stage 2: RL
RL_EPOCHS = 1
REWARD_MODE = "mwer"  # "mwer" | "wwer" | "llm" | "all"
REWARD_WEIGHT = 0.1   # Conservative (0.1), can tune to 0.3

# Paths
OUTPUT_DIR = "./nemo-afrispeech-output"
CHECKPOINT_DIR = "./checkpoints"
MANIFEST_DIR = "./manifests"

# Medical domain terms for WWER (AfriSpeech clinical vocabulary)
DOMAIN_TERMS = {
    # Common clinical terms in AfriSpeech
    "hypertension", "diabetes", "malaria", "fever", "headache",
    "medication", "prescription", "diagnosis", "symptoms", "patient",
    "doctor", "clinic", "hospital", "treatment", "blood", "pressure",
    "temperature", "pregnant", "pregnancy", "vaccine", "injection",
    # More specific medical terms
    "myocardial", "infarction", "tachycardia", "arrhythmia",
    "stethoscope", "auscultation", "echocardiogram", "tuberculosis",
    "antiretroviral", "paracetamol", "ibuprofen", "amoxicillin",
}
DOMAIN_TERM_WEIGHT = 3.0

print("Configuration loaded!")
print(f"  Model: {NEMO_MODEL_NAME}")
print(f"  SFT epochs: {SFT_EPOCHS}, RL epochs: {RL_EPOCHS}")
print(f"  Reward mode: {REWARD_MODE}, weight: {REWARD_WEIGHT}")

Configuration loaded!
  Model: stt_en_conformer_ctc_small
  SFT epochs: 2, RL epochs: 1
  Reward mode: mwer, weight: 0.1


# CELL 4: Dataset Loading Functions

In [3]:
def load_afrispeech_clinical(train_n: int, eval_n: int):
    """
    Load AfriSpeech-200 clinical subset via streaming.

    Dataset: ~120GB total, so we stream and filter to clinical domain only.
    Fields: 'transcript' (text), 'audio', 'domain' ('clinical' or 'general')
    """
    from datasets import Dataset, DatasetDict

    logger.info("Loading AfriSpeech-200 clinical subset ...")
    logger.info(f"  Requested: {train_n} train, {eval_n} eval samples")

    # Try intronhealth version first, fall back to tobiolatunji
    dataset_name = "intronhealth/afrispeech-200"

    try:
        ds_stream = load_dataset(
            dataset_name,
            "all",
            split="train",
            streaming=True,
        )
    except Exception as e:
        logger.warning(f"intronhealth failed: {e}, trying tobiolatunji...")
        dataset_name = "tobiolatunji/afrispeech-200"
        ds_stream = load_dataset(
            dataset_name,
            "all",
            split="train",
            streaming=True,
        )

    logger.info(f"  Using: {dataset_name}")

    # Filter to clinical domain
    train_samples = []
    eval_samples = []

    logger.info("  Filtering to clinical domain (may take 1-2 minutes)...")

    for sample in ds_stream:
        # Skip non-clinical
        if sample.get("domain", "").lower() != "clinical":
            continue

        # Skip empty transcripts
        if not sample.get("transcript", "").strip():
            continue

        # Collect samples
        if len(train_samples) < train_n:
            train_samples.append(sample)
        elif len(eval_samples) < eval_n:
            eval_samples.append(sample)
        else:
            break

        # Progress
        total = len(train_samples) + len(eval_samples)
        if total % 50 == 0:
            logger.info(f"    Collected {len(train_samples)} train, {len(eval_samples)} eval...")

    logger.info(f"  Final: {len(train_samples)} train, {len(eval_samples)} eval clinical samples")

    # Convert to Dataset objects
    train_ds = Dataset.from_list(train_samples)
    eval_ds = Dataset.from_list(eval_samples)

    # Resample audio to 16kHz (AfriSpeech is 44.1kHz)
    train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16_000))
    eval_ds = eval_ds.cast_column("audio", Audio(sampling_rate=16_000))

    ds = DatasetDict({
        "train": train_ds,
        "validation": eval_ds,
    })

    return ds, "transcript"  # AfriSpeech uses 'transcript' field


def load_librispeech(train_n: int, eval_n: int):
    """Fallback: Load LibriSpeech for testing."""
    logger.info("Loading LibriSpeech (fallback)...")

    try:
        ds = load_dataset(
            "librispeech_asr", "clean",
            split={
                "train": f"train.100[:{train_n}]",
                "validation": f"validation[:{eval_n}]",
            },
        )
    except Exception:
        ds = load_dataset(
            "librispeech_asr", "clean",
            split={
                "train": f"train.100[:{train_n}]",
                "validation": f"validation[:{eval_n}]",
            },
            trust_remote_code=True,
        )

    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    logger.info(f"  Train: {len(ds['train'])} | Eval: {len(ds['validation'])}")
    return ds, "text"


def load_dataset_with_fallback(train_n: int, eval_n: int):
    """Load AfriSpeech clinical, fall back to LibriSpeech if it fails."""
    try:
        return load_afrispeech_clinical(train_n, eval_n)
    except Exception as e:
        logger.warning(f"AfriSpeech failed ({e}). Using LibriSpeech fallback.")
        return load_librispeech(train_n, eval_n)

print("Dataset functions defined!")

Dataset functions defined!


# CELL 5: Manifest and Data Config Functions

In [4]:
def build_nemo_manifest(dataset, split_name: str, audio_dir: str, text_field: str = "transcript") -> str:
    """Convert HuggingFace dataset to NeMo manifest format."""
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(MANIFEST_DIR, exist_ok=True)

    manifest_path = os.path.join(MANIFEST_DIR, f"{split_name}_manifest.json")
    written = 0
    skipped = 0

    with open(manifest_path, "w") as f:
        for i, sample in enumerate(dataset):
            try:
                audio_array = sample["audio"]["array"]
                sr = sample["audio"]["sampling_rate"]
            except (KeyError, TypeError):
                skipped += 1
                continue

            text = sample.get(text_field, "").strip().lower()
            if not text:
                skipped += 1
                continue

            # Save audio
            wav_path = os.path.join(audio_dir, f"{split_name}_{i:05d}.wav")
            sf.write(wav_path, audio_array, sr)

            duration = len(audio_array) / sr

            # Skip very short or very long audio
            if duration < 0.5 or duration > 30.0:
                skipped += 1
                continue

            entry = {
                "audio_filepath": os.path.abspath(wav_path),
                "text": text,
                "duration": round(duration, 3),
            }
            f.write(json.dumps(entry) + "\n")
            written += 1

    logger.info(f"  Manifest: {manifest_path} ({written} written, {skipped} skipped)")
    return manifest_path

print("Manifest function defined!")

Manifest function defined!


In [15]:
def build_data_config(train_manifest: str, val_manifest: str) -> DictConfig:
    """Build NeMo data configuration."""
    return OmegaConf.create({
        "train_ds": {
            "manifest_filepath": train_manifest,
            "sample_rate": 16000,
            "batch_size": BATCH_SIZE,
            "shuffle": True,
            "num_workers": 4,
            "pin_memory": True,
            "trim_silence": False,
            "max_duration": 20.0,
            "min_duration": 0.1,
        },
        "validation_ds": {
            "manifest_filepath": val_manifest,
            "sample_rate": 16000,
            "batch_size": BATCH_SIZE,
            "shuffle": False,
            "num_workers": 4,
            "pin_memory": True,
        },
    })

print("Manifest functions defined!")

Manifest functions defined!


# CELL 6: Reward Functions

In [5]:
def compute_mwer_reward(hypotheses: List[str], references: List[str]) -> torch.Tensor:
    """MWER reward: reward = 1 - WER"""
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not ref.strip():
            rewards.append(0.5)
            continue
        error_rate = compute_wer_jiwer(ref, hyp)
        reward = max(0.0, 1.0 - error_rate)
        rewards.append(reward)
    return torch.tensor(rewards, dtype=torch.float32)


def compute_wwer_reward(hypotheses: List[str], references: List[str]) -> torch.Tensor:
    """Weighted WER reward: domain terms weighted higher."""
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not ref.strip():
            rewards.append(0.5)
            continue

        ref_words = ref.lower().split()
        hyp_words = hyp.lower().split()

        weights = [DOMAIN_TERM_WEIGHT if w in DOMAIN_TERMS else 1.0 for w in ref_words]
        total_weight = sum(weights)

        errors = 0.0
        for i, ref_w in enumerate(ref_words):
            if i >= len(hyp_words) or hyp_words[i] != ref_w:
                errors += weights[i]

        if len(hyp_words) > len(ref_words):
            errors += len(hyp_words) - len(ref_words)
            total_weight += len(hyp_words) - len(ref_words)

        weighted_error = errors / total_weight if total_weight > 0 else 1.0
        reward = max(0.0, 1.0 - weighted_error)
        rewards.append(reward)

    return torch.tensor(rewards, dtype=torch.float32)


def compute_llm_reward(hypotheses: List[str], references: List[str], use_mock: bool = True) -> torch.Tensor:
    """LLM-based reward (mock for testing)."""
    base_rewards = compute_mwer_reward(hypotheses, references)
    noise = torch.randn_like(base_rewards) * 0.05
    return torch.clamp(base_rewards + noise, 0.0, 1.0)


def compute_combined_reward(hypotheses: List[str], references: List[str]) -> torch.Tensor:
    """Combine all reward signals."""
    r_mwer = compute_mwer_reward(hypotheses, references)
    r_wwer = compute_wwer_reward(hypotheses, references)
    r_llm = compute_llm_reward(hypotheses, references, use_mock=True)
    return (r_mwer + r_wwer + r_llm) / 3.0

print("Reward functions defined!")

Reward functions defined!


# CELL 7: Evaluation Function


In [6]:
def evaluate_wer(model, manifest_path: str) -> float:
    """Evaluate WER on a manifest file."""
    references = []
    audio_paths = []

    with open(manifest_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            references.append(entry["text"])
            audio_paths.append(entry["audio_filepath"])

    hypotheses = model.transcribe(audio_paths, batch_size=BATCH_SIZE)

    hyps_text = []
    for h in hypotheses:
        if isinstance(h, str):
            hyps_text.append(h)
        elif hasattr(h, 'text'):
            hyps_text.append(h.text)
        else:
            hyps_text.append(str(h))

    wer = compute_wer_jiwer(references, hyps_text) * 100
    logger.info(f"  WER: {wer:.2f}%")
    return wer

print("Evaluation function defined!")

Evaluation function defined!


# CELL 8: Model Loading Functions


In [7]:
def load_model_for_sft(model_name: str) -> EncDecCTCModelBPE:
    """Load model for SFT (no reward injection)."""
    logger.info(f"Loading model for SFT: {model_name}")
    model = EncDecCTCModelBPE.from_pretrained(model_name)
    model.reward_weight = 0.0
    model._step_logs = []
    return model


def load_model_for_rl(checkpoint_path: str, reward_mode: str = "mwer", reward_weight: float = 0.1) -> EncDecCTCModelBPE:
    """Load model from SFT checkpoint with reward injection for RL."""
    logger.info(f"Loading model for RL from: {checkpoint_path}")
    logger.info(f"  Reward mode: {reward_mode}, weight: {reward_weight}")

    model = EncDecCTCModelBPE.restore_from(checkpoint_path)
    model.reward_mode = reward_mode
    model.reward_weight = reward_weight
    model._step_logs = []

    original_training_step = model.training_step

    def patched_training_step(self, batch, batch_nb):
        result = original_training_step(batch, batch_nb)

        if self.reward_weight == 0.0:
            return result

        if isinstance(result, dict):
            ctc_loss = result['loss']
        else:
            ctc_loss = result

        # Get batch data
        if hasattr(batch, 'audio'):
            signal, signal_len = batch.audio, batch.audio_lens
            transcript, transcript_len = batch.tokens, batch.token_lens
        elif hasattr(batch, 'input_signal'):
            signal, signal_len = batch.input_signal, batch.input_signal_length
            transcript, transcript_len = batch.targets, batch.target_lengths
        else:
            signal, signal_len, transcript, transcript_len = batch[:4]

        with torch.no_grad():
            log_probs, encoded_len, _ = self.forward(input_signal=signal, input_signal_length=signal_len)

            hyps_raw = self.wer.decoding.ctc_decoder_predictions_tensor(
                decoder_outputs=log_probs,
                decoder_lengths=encoded_len,
                return_hypotheses=False,
            )

            hyps = [h.text if hasattr(h, 'text') else str(h) if not isinstance(h, str) else h for h in hyps_raw]

            refs = []
            for i in range(transcript.size(0)):
                target_len = transcript_len[i].item()
                target_ids = transcript[i, :target_len].tolist()
                refs.append(self.tokenizer.ids_to_text(target_ids))

        # Compute reward
        if self.reward_mode == "mwer":
            rewards = compute_mwer_reward(hyps, refs)
        elif self.reward_mode == "wwer":
            rewards = compute_wwer_reward(hyps, refs)
        elif self.reward_mode == "llm":
            rewards = compute_llm_reward(hyps, refs, use_mock=True)
        elif self.reward_mode == "all":
            rewards = compute_combined_reward(hyps, refs)
        else:
            rewards = compute_mwer_reward(hyps, refs)

        rewards = rewards.to(ctc_loss.device)
        penalty = 1.0 - rewards.mean()
        total_loss = ctc_loss + self.reward_weight * penalty

        self._step_logs.append({
            "step": batch_nb,
            "ctc_loss": ctc_loss.item(),
            "reward_mean": rewards.mean().item(),
            "penalty": penalty.item(),
            "total_loss": total_loss.item(),
        })

        if isinstance(result, dict):
            result['loss'] = total_loss
        else:
            result = total_loss

        return result

    model.training_step = types.MethodType(patched_training_step, model)
    logger.info("  Reward injection enabled")
    return model

print("Model loading functions defined!")

Model loading functions defined!


# CELL 9: Training Helper Functions


In [8]:
def compute_warmup_steps(num_samples: int, batch_size: int, num_epochs: int) -> int:
    """Compute warmup: 10% of total steps, clamped to [10, 500]."""
    steps_per_epoch = math.ceil(num_samples / batch_size)
    total_steps = steps_per_epoch * num_epochs
    warmup = max(10, min(500, int(total_steps * 0.1)))
    logger.info(f"  Warmup: {warmup} steps (total steps: {total_steps})")
    return warmup


def create_trainer(num_epochs: int, warmup_steps: int, learning_rate: float = None):
    """Create Lightning trainer and optimizer config."""
    if learning_rate is None:
        learning_rate = LEARNING_RATE

    trainer = pl.Trainer(
        devices=1,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        max_epochs=num_epochs,
        precision="16-mixed",
        log_every_n_steps=10,
        val_check_interval=1.0,
        enable_checkpointing=False,
        logger=False,
    )

    optim_config = OmegaConf.create({
        "name": "adamw",
        "lr": learning_rate,
        "weight_decay": 1e-3,
        "sched": {
            "name": "CosineAnnealing",
            "warmup_steps": warmup_steps,
            "min_lr": 1e-6,
        },
    })

    return trainer, optim_config

print("Training helpers defined!")

Training helpers defined!


# CELL 10: Stage 1 - SFT Training


In [9]:
def run_sft_stage(train_manifest: str, val_manifest: str, save_path: str):
    """Stage 1: Supervised Fine-Tuning (no reward)."""
    logger.info("=" * 60)
    logger.info("STAGE 1: SUPERVISED FINE-TUNING (SFT)")
    logger.info("=" * 60)

    # Load fresh pretrained model
    model = load_model_for_sft(NEMO_MODEL_NAME)

    # Setup data
    data_cfg = build_data_config(train_manifest, val_manifest)
    model.setup_training_data(data_cfg.train_ds)
    model.setup_validation_data(data_cfg.validation_ds)

    # Compute warmup based on actual data size
    num_train_samples = sum(1 for _ in open(train_manifest))
    warmup_steps = compute_warmup_steps(num_train_samples, BATCH_SIZE, SFT_EPOCHS)

    # Create trainer
    trainer, optim_config = create_trainer(SFT_EPOCHS, warmup_steps)
    model.set_trainer(trainer)
    model.setup_optimization(optim_config)

    # Train
    logger.info("Starting SFT training...")
    start = time.time()
    trainer.fit(model)
    elapsed = time.time() - start
    logger.info(f"SFT complete! Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # Save checkpoint
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model.save_to(save_path)
    logger.info(f"SFT model saved: {save_path}")

    # Evaluate
    final_wer = evaluate_wer(model, val_manifest)
    logger.info(f"SFT Final WER: {final_wer:.2f}%")

    return model, final_wer

print("SFT stage function defined!")

SFT stage function defined!


# CELL 11: Stage 2 - RL Training


In [10]:
def run_rl_stage(sft_checkpoint: str, train_manifest: str, val_manifest: str, save_path: str):
    """Stage 2: RL Fine-Tuning on SFT checkpoint."""
    logger.info("=" * 60)
    logger.info("STAGE 2: REINFORCEMENT LEARNING (RL)")
    logger.info(f"  Loading SFT checkpoint: {sft_checkpoint}")
    logger.info(f"  Reward mode: {REWARD_MODE}, weight: {REWARD_WEIGHT}")
    logger.info("=" * 60)

    # Load from SFT checkpoint with reward injection
    model = load_model_for_rl(sft_checkpoint, REWARD_MODE, REWARD_WEIGHT)

    # Setup data
    data_cfg = build_data_config(train_manifest, val_manifest)
    model.setup_training_data(data_cfg.train_ds)
    model.setup_validation_data(data_cfg.validation_ds)

    # Compute warmup
    num_train_samples = sum(1 for _ in open(train_manifest))
    warmup_steps = compute_warmup_steps(num_train_samples, BATCH_SIZE, RL_EPOCHS)

    # Create trainer with LOWER learning rate for RL
    trainer, optim_config = create_trainer(RL_EPOCHS, warmup_steps, LEARNING_RATE * 0.5)
    model.set_trainer(trainer)
    model.setup_optimization(optim_config)

    # Train
    logger.info("Starting RL training...")
    start = time.time()
    trainer.fit(model)
    elapsed = time.time() - start
    logger.info(f"RL complete! Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # Save checkpoint
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model.save_to(save_path)
    logger.info(f"RL model saved: {save_path}")

    # Evaluate
    final_wer = evaluate_wer(model, val_manifest)
    logger.info(f"RL Final WER: {final_wer:.2f}%")

    # Print reward statistics
    if model._step_logs:
        df = pd.DataFrame(model._step_logs)
        logger.info("\nRL Training Statistics:")
        logger.info(f"  Mean reward: {df['reward_mean'].mean():.3f}")
        logger.info(f"  Mean penalty: {df['penalty'].mean():.3f}")
        logger.info(f"  CTC loss: {df['ctc_loss'].iloc[0]:.3f} → {df['ctc_loss'].iloc[-1]:.3f}")

    return model, final_wer

print("RL stage function defined!")

RL stage function defined!


# CELL 12: Prepare Data


In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

print("Loading AfriSpeech-200 clinical dataset...")
print("(This streams from HuggingFace and filters to clinical domain - may take 1-2 min)")
print()

dataset, text_field = load_dataset_with_fallback(TRAIN_SAMPLES, EVAL_SAMPLES)

print("\nBuilding NeMo manifests...")
audio_dir = os.path.join(OUTPUT_DIR, "audio")
train_manifest = build_nemo_manifest(dataset["train"], "train", audio_dir, text_field)
val_manifest = build_nemo_manifest(dataset["validation"], "val", audio_dir, text_field)

print(f"\n{'='*60}")
print("DATA READY!")
print(f"  Train manifest: {train_manifest}")
print(f"  Val manifest: {val_manifest}")
print(f"{'='*60}")

Loading AfriSpeech-200 clinical dataset...
(This streams from HuggingFace and filters to clinical domain - may take 1-2 min)



README.md: 0.00B [00:00, ?B/s]

afrispeech-200.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

afrispeech-200.py: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/2620 [00:00<?, ? examples/s]

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]


Building NeMo manifests...

DATA READY!
  Train manifest: ./manifests/train_manifest.json
  Val manifest: ./manifests/val_manifest.json


# CELL 13: Run Stage 1 - SFT


In [16]:
sft_checkpoint = os.path.join(CHECKPOINT_DIR, "sft_model.nemo")
sft_model, sft_wer = run_sft_stage(train_manifest, val_manifest, sft_checkpoint)

print(f"\n{'='*60}")
print(f"SFT COMPLETE!")
print(f"  WER: {sft_wer:.2f}%")
print(f"  Checkpoint: {sft_checkpoint}")
print(f"{'='*60}")

[NeMo I 2026-03-31 23:39:04 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
[NeMo I 2026-03-31 23:39:04 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo
[NeMo I 2026-03-31 23:39:04 common:939] Instantiating model from pre-trained checkpoint
[NeMo I 2026-03-31 23:39:05 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-31 23:39:06 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-03-31 23:39:06 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-03-31 23:39:06 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
[NeMo I 2026-03-31 23:39:06 collections:201] Dataset loaded with 200 files totalling 0.74 hours
[NeMo I 2026-03-31 23:39:06 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-03-31 23:39:06 collections:201] Dataset loaded with 100 files totalling 0.14 hours
[NeMo I 2026-03-31 23:39:06 collections:202] 0 files were filtered totalling 0.00 hours


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


[NeMo I 2026-03-31 23:39:06 modelPT:830] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.0001
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-03-31 23:39:06 lr_scheduler:995] Scheduler "<nemo.core.optim.lr_scheduler.CosineAnnealing object at 0x79245036b830>" 
    will be used during training (effective maximum steps = 50) - 
    Parameters : 
    (warmup_steps: 10
    min_lr: 1.0e-06
    max_steps: 50
    )


...

[NeMo I 2026-03-31 23:39:32 wer:336] 
    
[NeMo I 2026-03-31 23:39:32 wer:337] WER reference:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 23:39:32 wer:338] WER predicted:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 23:39:32 wer:336] 
    
[NeMo I 2026-03-31 23:39:32 wer:337] WER reference:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:39:32 wer:338] WER predicted:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:39:32 wer:336] 
    
[NeMo I 2026-03-31 23:39:32 wer:337] WER reference:fortunately there was nothing from his wife either
[NeMo I 2026-03-31 23:39:32 wer:338] WER 

Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-03-31 23:39:46 wer:336] 
    
[NeMo I 2026-03-31 23:39:46 wer:337] WER reference:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 23:39:46 wer:338] WER predicted:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 23:39:46 wer:336] 
    
[NeMo I 2026-03-31 23:39:46 wer:337] WER reference:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:39:46 wer:338] WER predicted:for some reason he felt that if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:39:46 wer:336] 
    
[NeMo I 2026-03-31 23:39:46 wer:337] WER reference:fortunately there was nothing from his wife either
[NeMo I 2026-03-31 23:39:46 wer:338] WE

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.
[NeMo W 2026-03-31 23:39:47 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-31 23:39:47 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 13it [00:08,  1.57it/s]


SFT COMPLETE!
  WER: 8.75%
  Checkpoint: ./checkpoints/sft_model.nemo


# CELL 14: Run Stage 2 - RL


In [17]:
rl_checkpoint = os.path.join(CHECKPOINT_DIR, "rl_model.nemo")
rl_model, rl_wer = run_rl_stage(sft_checkpoint, train_manifest, val_manifest, rl_checkpoint)

print(f"\n{'='*60}")
print(f"RL COMPLETE!")
print(f"  WER: {rl_wer:.2f}%")
print(f"  Checkpoint: {rl_checkpoint}")
print(f"{'='*60}")

[NeMo I 2026-03-31 23:40:05 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-31 23:40:05 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: ./manifests/train_manifest.json
    sample_rate: 16000
    batch_size: 8
    shuffle: true
    num_workers: 4
    pin_memory: true
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    
[NeMo W 2026-03-31 23:40:05 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: ./manifests/val_manifest.json
    sample_rate: 16000
    batch_size: 8
    shuffle: false
    num_workers: 4
    pin_memory: true
    
[NeMo W 2026-03-31 23:40:05 modelPT:202] Please call the ModelPT.setup_test_data() or ModelPT.setup_multipl

[NeMo I 2026-03-31 23:40:05 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /content/checkpoints/sft_model.nemo.
[NeMo I 2026-03-31 23:40:05 collections:201] Dataset loaded with 200 files totalling 0.74 hours
[NeMo I 2026-03-31 23:40:05 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-03-31 23:40:06 collections:201] Dataset loaded with 100 files totalling 0.14 hours
[NeMo I 2026-03-31 23:40:06 collections:202] 0 files were filtered totalling 0.00 hours


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


[NeMo I 2026-03-31 23:40:06 modelPT:830] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 5e-05
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-03-31 23:40:06 lr_scheduler:995] Scheduler "<nemo.core.optim.lr_scheduler.CosineAnnealing object at 0x7923ac90ea80>" 
    will be used during training (effective maximum steps = 25) - 
    Parameters : 
    (warmup_steps: 10
    min_lr: 1.0e-06
    max_steps: 25
    )


...

[NeMo I 2026-03-31 23:40:10 wer:336] 
    
[NeMo I 2026-03-31 23:40:10 wer:337] WER reference:bluff caught him by the arm wait just a minute or two will he pleaded tell us some more where did all this happen frank knows where that squirrel colony have their nest in the tree that's got a hole in the trunk about thirty feet up the other replied
[NeMo I 2026-03-31 23:40:10 wer:338] WER predicted:bff cught him by the arm just a minute or to will he pleaded tell us some more where did all this happen frank knows whether coloy have their nest in the tree thats got a whole about thirty feet up the other replied
[NeMo I 2026-03-31 23:40:14 wer:336] 
    
[NeMo I 2026-03-31 23:40:14 wer:337] WER reference:but you're away off in your guess i've so longed to meet up with one when i had my camera with me that i've been picturing how he'd look and frank believe me it was a beaut a regular monster how did it happen will asked frank
[NeMo I 2026-03-31 23:40:14 wer:338] WER predicted:but you're a away

Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-03-31 23:40:16 wer:336] 
    
[NeMo I 2026-03-31 23:40:16 wer:337] WER reference:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 23:40:16 wer:338] WER predicted:he was in a fevered state of mind owing to the blight his wife's action threatened to cast up on his entire future
[NeMo I 2026-03-31 23:40:16 wer:336] 
    
[NeMo I 2026-03-31 23:40:16 wer:337] WER reference:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:40:16 wer:338] WER predicted:some reason he felt that if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 23:40:16 wer:336] 
    
[NeMo I 2026-03-31 23:40:16 wer:337] WER reference:fortunately there was nothing from his wife either
[NeMo I 2026-03-31 23:40:16 wer:338] WER p

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.
[NeMo W 2026-03-31 23:40:18 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-31 23:40:18 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 13it [00:06,  1.97it/s]


RL COMPLETE!
  WER: 9.57%
  Checkpoint: ./checkpoints/rl_model.nemo


# CELL 15: Summary


In [18]:
print("\n" + "=" * 60)
print("TRAINING COMPLETE - AFRISPEECH-200 CLINICAL")
print("=" * 60)
print(f"  SFT WER: {sft_wer:.2f}%")
print(f"  RL WER:  {rl_wer:.2f}%")
print(f"  Change:  {rl_wer - sft_wer:+.2f}%")
print(f"\nReward config:")
print(f"  Mode: {REWARD_MODE}, Weight: {REWARD_WEIGHT}")
print(f"\nCheckpoints saved:")
print(f"  SFT: {sft_checkpoint}")
print(f"  RL:  {rl_checkpoint}")
print("=" * 60)

# Show RL training logs
if hasattr(rl_model, '_step_logs') and rl_model._step_logs:
    print("\nRL Step Logs (first 5):")
    df = pd.DataFrame(rl_model._step_logs)
    print(df.head().to_string())


TRAINING COMPLETE - AFRISPEECH-200 CLINICAL
  SFT WER: 8.75%
  RL WER:  9.57%
  Change:  +0.83%

Reward config:
  Mode: mwer, Weight: 0.1

Checkpoints saved:
  SFT: ./checkpoints/sft_model.nemo
  RL:  ./checkpoints/rl_model.nemo

RL Step Logs (first 5):
   step   ctc_loss  reward_mean   penalty  total_loss
0     0  34.689644     0.824084  0.175916   34.707237
1     1  43.709068     0.793172  0.206828   43.729752
2     2  41.889702     0.865486  0.134514   41.903152
3     3  35.584732     0.884520  0.115480   35.596279
4     4  42.952534     0.824492  0.175508   42.970085
